# FlavourBench release checkpoint — 9 August 2026

## tl;dr

This notebook validates the exact counts and content-addressed inputs used by the release-checkpoint report. The retrospective pilot and paper package are real and reproducible, but the current benchmark remains **NO-GO**: current quality judgments = 0, admissible human judgments = 0, the Qwen-containing K16 alternative is offline and unranked, statistical release gates are not validated, and human/recovery gates remain fail-closed.

## Context & Methods

The unit of validation is the frozen artifact or source file, not a live provider response. The notebook recomputes physical and embedded semantic SHA-256 digests, reconciles release-gate totals, checks candidate-roster uniqueness and claim flags, verifies human-workload arithmetic and the synchronized v3 checksum package, audits K16 simulation identities across append-only predecessors, and inspects the compiled PDF metadata.

### Key Assumptions

- The 2 August scorecard remains the controlling frozen gate-count source until an additive successor exists.
- Route compatibility is not treated as a quality observation.
- Historical artifacts are not rewritten when newer findings supersede them.
- The two flawed K16 resolution artifacts remain append-only and not for use. The accepted successor uses exactly 100 unseen identities at indices 140 through 239 and remains development-only NO-GO.
- The routed 14, paper/public operational 16, and blocked offline K16 are distinct panels.

## Data

All paths are repository-relative and contain no credentials or raw participant data.

In [1]:
from __future__ import annotations

import hashlib
import json
import subprocess
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'governance').is_dir() and (candidate / 'flavourbench').is_dir():
            return candidate
    raise RuntimeError('evaluation-paper repository root not found')


ROOT = find_repo_root(Path.cwd().resolve())
SCORECARD = ROOT / 'governance/reviews/FLAVOURBENCH-MULTI-DIMENSION-GOVERNANCE-SCORES-2026-08-02.json'
K14_POWER = ROOT / 'flavourbench/artifacts/season1/sampling-power-validation-v1-candidate/sampling-power-validation-v1-candidate-1077a8c237854c9ba96d7b03f22aa60e0562c5f8aeb986fd52e5e167a9694504.json'
K16_PANEL = ROOT / 'flavourbench/artifacts/season1/study-design-16-model-alternative-v1-candidate/study-design-16-model-alternative-v1-candidate-675cdb81bcbd54cf3532025ae70069723d7e9843b0eeeb92f1ea38bee7c58278.json'
K16_POWER_PREDECESSOR = ROOT / 'flavourbench/artifacts/season1/full-k16-arena-resolution-audit-v1-candidate/full-k16-arena-resolution-audit-v1-candidate-b59dfc07280f972b83e00c5699a0f28e3da61135f2da22cfaf4638a4d1391910.json'
K16_POWER_OVERLAP = ROOT / 'flavourbench/artifacts/season1/full-k16-arena-resolution-audit-v1-candidate/full-k16-arena-resolution-audit-v1-candidate-64f9f2f8afad51ded9f3be8b84d0c91c0259eae78c5b1086099dbb9f4f26eb1a.json'
K16_POWER_FINAL = ROOT / 'flavourbench/artifacts/season1/full-k16-arena-resolution-audit-v1-candidate/full-k16-arena-resolution-audit-v1-candidate-596ef6cd38132a351605ce0f734489262e372e1c148337a88238dbb466a12ddc.json'
ACTIVATION = ROOT / 'protocol/human-study/human-study-activation-current-v1.json'
HUMAN_PACKAGE = ROOT / 'protocol/human-study/HUMAN-STUDY-GO-PACKAGE-v3.sha256'
PDF = ROOT / 'paper/flavourbench/build/flavourbench.pdf'
print({'repository_root_detected': True, 'root_name': ROOT.name})

{'repository_root_detected': True, 'root_name': 'evaluation-paper'}


In [2]:
def physical_sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def validate_content_addressed_json(path: Path) -> dict:
    payload = json.loads(path.read_bytes())
    stated = payload.pop('artifact_sha256')
    canonical = json.dumps(
        payload, sort_keys=True, separators=(',', ':'), ensure_ascii=False
    ).encode()
    semantic = hashlib.sha256(canonical).hexdigest()
    assert semantic == stated
    return {
        'physical_sha256': physical_sha256(path),
        'semantic_sha256': semantic,
        'payload': payload,
    }


k14 = validate_content_addressed_json(K14_POWER)
k16 = validate_content_addressed_json(K16_PANEL)
k16_power_predecessor = validate_content_addressed_json(K16_POWER_PREDECESSOR)
k16_power_overlap = validate_content_addressed_json(K16_POWER_OVERLAP)
k16_power_final = validate_content_addressed_json(K16_POWER_FINAL)
assert k14['semantic_sha256'] == K14_POWER.stem.rsplit('-', 1)[-1]
assert k16['semantic_sha256'] == K16_PANEL.stem.rsplit('-', 1)[-1]
assert k16_power_predecessor['semantic_sha256'] == K16_POWER_PREDECESSOR.stem.rsplit('-', 1)[-1]
assert k16_power_overlap['semantic_sha256'] == K16_POWER_OVERLAP.stem.rsplit('-', 1)[-1]
assert k16_power_final['semantic_sha256'] == K16_POWER_FINAL.stem.rsplit('-', 1)[-1]
print(json.dumps({
    'k14_power': {k: v for k, v in k14.items() if k != 'payload'},
    'k16_panel': {k: v for k, v in k16.items() if k != 'payload'},
    'k16_power_predecessor': {k: v for k, v in k16_power_predecessor.items() if k != 'payload'},
    'k16_power_overlap': {k: v for k, v in k16_power_overlap.items() if k != 'payload'},
    'k16_power_final': {k: v for k, v in k16_power_final.items() if k != 'payload'},
}, indent=2))

{
  "k14_power": {
    "physical_sha256": "140419f71a43c0276e87350ad367e6cf2e896879d02ebfaf9c1131a39d1202f9",
    "semantic_sha256": "1077a8c237854c9ba96d7b03f22aa60e0562c5f8aeb986fd52e5e167a9694504"
  },
  "k16_panel": {
    "physical_sha256": "f26bac6aa12a6dfca4ab0ee8ff7f2a0814a2214e8f21f9292d09221ec5104740",
    "semantic_sha256": "675cdb81bcbd54cf3532025ae70069723d7e9843b0eeeb92f1ea38bee7c58278"
  },
  "k16_power_predecessor": {
    "physical_sha256": "82caec6dba223c089b972775236ea47e5ef380b1beffd437461b6160b58ff893",
    "semantic_sha256": "b59dfc07280f972b83e00c5699a0f28e3da61135f2da22cfaf4638a4d1391910"
  },
  "k16_power_overlap": {
    "physical_sha256": "5685a70d168ba6dde901eb5481e80e9ad3574eb3e1de4131f84c4e53dff23920",
    "semantic_sha256": "64f9f2f8afad51ded9f3be8b84d0c91c0259eae78c5b1086099dbb9f4f26eb1a"
  },
  "k16_power_final": {
    "physical_sha256": "b07fa02b5633c13d331ad5490c150a514f92aa4bb7abdce985b156aa9e8af17c",
    "semantic_sha256": "596ef6cd38132a351605ce0f7344

## Results

In [3]:
scorecard = json.loads(SCORECARD.read_text())
gate_breakdown = {
    'full_pass': 4,
    'catalog_only': 1,
    'partial': 1,
    'not_started': 5,
    'blocked': 18,
}
assert sum(gate_breakdown.values()) == 29
assert scorecard['evidence_counts']['release_gates'] == 29
assert scorecard['evidence_counts']['release_gates_full_pass'] == 4
assert scorecard['evidence_counts']['release_gates_blocked'] == 18
gate_breakdown

{'full_pass': 4,
 'catalog_only': 1,
 'partial': 1,
 'not_started': 5,
 'blocked': 18}

In [4]:
panel = k16['payload']
model_ids = [row['model_id'] for row in panel['candidate_model_panel']['models']]
assert len(model_ids) == len(set(model_ids)) == 16
assert model_ids.count('qwen3.8-max') == 1
assert model_ids.count('moonshotai/kimi-k3') == 1
claim_boundary = panel['claim_boundary']
assert claim_boundary['official'] is False
assert claim_boundary['rank_eligible'] is False
assert claim_boundary['quality_observations_used'] == 0
assert claim_boundary['human_judgments_made_by_this_artifact'] == 0
{
    'model_count': len(model_ids),
    'unique_model_count': len(set(model_ids)),
    'qwen_count': model_ids.count('qwen3.8-max'),
    'kimi_count': model_ids.count('moonshotai/kimi-k3'),
    'official': claim_boundary['official'],
    'rank_eligible': claim_boundary['rank_eligible'],
    'quality_observations': claim_boundary['quality_observations_used'],
}

{'model_count': 16,
 'unique_model_count': 16,
 'qwen_count': 1,
 'kimi_count': 1,
 'official': False,
 'rank_eligible': False,
 'quality_observations': 0}

In [5]:
activation = json.loads(ACTIVATION.read_text())
primary_judgments = (800 + 800) * 2
repeat_presentations = 400
assert primary_judgments == 3_200
assert primary_judgments + repeat_presentations == 3_600
assert activation['status'] == 'blocked'
assert len(activation['blockers']) == 10

package_entries = []
for line in HUMAN_PACKAGE.read_text().splitlines():
    expected, relative = line.split(maxsplit=1)
    member = (ROOT / relative).resolve()
    assert member.is_file()
    assert physical_sha256(member) == expected
    package_entries.append(relative)
assert len(package_entries) == 61
{
    'primary_judgments': primary_judgments,
    'repeat_presentations': repeat_presentations,
    'total_presentations': primary_judgments + repeat_presentations,
    'activation_status': activation['status'],
    'activation_blockers': len(activation['blockers']),
    'human_package_entries_verified': len(package_entries),
    'human_package_sha256': physical_sha256(HUMAN_PACKAGE),
}

{'primary_judgments': 3200,
 'repeat_presentations': 400,
 'total_presentations': 3600,
 'activation_status': 'blocked',
 'activation_blockers': 10,
 'human_package_entries_verified': 61,
 'human_package_sha256': '7273a74e1e75a2d2943cf7eef1fbc8d63d078cc1bfc2c19ff9a55e3eb4a78a82'}

In [6]:
checks = k14['payload']['prespecified_candidate_acceptance']['checks']
failed_core_checks = [row['check_id'] for row in checks if row['core'] and not row['passed']]
assert len(failed_core_checks) == 12

def simulation_identity(row: dict) -> tuple:
    return (
        row['layout_sha256'], row['config_id'], row['shift_elo'],
        row['dataset_index'], row['dataset_seed'],
    )

predecessor_p80 = {
    simulation_identity(row)
    for row in k16_power_predecessor['payload']['dataset_records']
    if row['config_id'] == 'p80_r2' and row['shift_elo'] == 50.0
}
overlap_confirmation = {
    simulation_identity(row)
    for row in k16_power_overlap['payload']['dataset_records']
    if row.get('record_lineage') == 'generated_disjoint_successor_indices_40_139'
}
identity_overlap = predecessor_p80 & overlap_confirmation
assert len(predecessor_p80) == 100
assert len(overlap_confirmation) == 100
assert len(identity_overlap) == 60
assert {identity[3] for identity in identity_overlap} == set(range(40, 100))
assert k16_power_overlap['payload']['decision']['overall_verdict'] == 'NO-GO'

prior_identity_union = {
    simulation_identity(row)
    for artifact in (k16_power_predecessor, k16_power_overlap)
    for row in artifact['payload']['dataset_records']
}
fresh_records = [
    row for row in k16_power_final['payload']['dataset_records']
    if row.get('record_lineage') == 'generated_fresh_successor_indices_140_239'
]
fresh_identities = {simulation_identity(row) for row in fresh_records}
assert len(fresh_records) == len(fresh_identities) == 100
assert {row['dataset_index'] for row in fresh_records} == set(range(140, 240))
assert fresh_identities.isdisjoint(prior_identity_union)
assert k16_power_final['payload']['decision']['overall_verdict'] == 'NO-GO'
assert k16_power_final['payload']['decision']['selected_final_frozen_marginal_power_lower_bound'] == 0.914823947
assert k16_power_final['payload']['decision']['selected_final_simultaneous_top_lower_bound'] == 0.568827249
assert k16_power_final['payload']['decision']['frozen_50_elo_marginal_target_passed'] is True
assert k16_power_final['payload']['decision']['simultaneous_top_resolution_passed'] is False
assert k16_power_final['payload']['claim_boundary']['development_only'] is True
assert not any(
    value for key, value in k16_power_final['payload']['claim_boundary'].items()
    if key != 'development_only'
)

pdfinfo = subprocess.run(
    ['pdfinfo', str(PDF)], check=True, capture_output=True, text=True
).stdout
pdf_fields = dict(
    line.split(':', 1) for line in pdfinfo.splitlines() if ':' in line
)
pages = int(pdf_fields['Pages'].strip())
assert pages == 25
{
    'k14_failed_core_checks': len(failed_core_checks),
    'failed_check_ids': failed_core_checks,
    'k16_predecessor_unique_p80_identities': len(predecessor_p80),
    'k16_purported_successor_confirmation_identities': len(overlap_confirmation),
    'k16_repeated_simulation_identities': len(identity_overlap),
    'k16_fresh_record_count': len(fresh_records),
    'k16_fresh_identity_overlap': len(fresh_identities & prior_identity_union),
    'k16_marginal_lower_bound': k16_power_final['payload']['decision']['selected_final_frozen_marginal_power_lower_bound'],
    'k16_simultaneous_top_lower_bound': k16_power_final['payload']['decision']['selected_final_simultaneous_top_lower_bound'],
    'k16_provenance_status': 'fresh_140_239_verified_development_only_no_go',
    'pdf_pages': pages,
    'pdf_bytes': PDF.stat().st_size,
    'pdf_sha256': physical_sha256(PDF),
}

{'k14_failed_core_checks': 12,
 'failed_check_ids': ['plausible_rater_dropout_overall_coverage',
  'outcome_dependent_missingness_overall_coverage',
  'outcome_dependent_missingness_absolute_bias',
  'calibrated_0_08_complete_family_power_at_0_08',
  'calibrated_0_08_complete_per_model_power',
  'calibrated_0_08_complete_per_model_precision',
  'calibrated_0_08_complete_arena_proxy_top_identification_at_50_elo',
  'calibrated_0_08_high_dependence_family_power_at_0_08',
  'calibrated_0_08_high_dependence_per_model_power',
  'calibrated_0_08_high_dependence_per_model_precision',
  'calibrated_0_08_high_dependence_arena_proxy_top_identification_at_50_elo',
  'reliability_track_precision'],
 'k16_predecessor_unique_p80_identities': 100,
 'k16_purported_successor_confirmation_identities': 100,
 'k16_repeated_simulation_identities': 60,
 'k16_fresh_record_count': 100,
 'k16_fresh_identity_overlap': 0,
 'k16_marginal_lower_bound': 0.914823947,
 'k16_simultaneous_top_lower_bound': 0.568827249,

## Takeaways

- The frozen release-gate breakdown is internally consistent and remains dominated by blocked gates.
- The blocked offline K16 roster is uniquely defined and honestly fail-closed; it is not the paper/public operational 16.
- Human workload arithmetic is consistent, and the inactive v3 package verifies all 61 current files; ten governance blockers still prohibit activation.
- The corrected K14 design fails 12 core checks, so it cannot support current official ranks.
- The corrected P80×R2 K16 confirmation has 100 unseen identities. Its marginal lower bound is 0.914823947, but simultaneous top-model support has lower bound 0.568827249, so the result remains development-only NO-GO.
- The rebuilt 25-page paper incorporates the August 9 human- and statistical-readiness successors, supports a retrospective audit claim, and is not an official current-model leaderboard.